# 面试题：Saga 补偿事务怎样处理长链路副作用？

可复述答案：跨服务流程拆成可独立提交的本地事务，并为每步定义补偿、幂等键、后置验证与不可逆边界。后续失败时逆序补偿已成功的可逆动作；补偿失败进入明确 pending 队列，不能悄悄标成功。Saga 不等于数据库事务，已发送通知通常只能发送更正而不能撤回。

## 真实案例

出差预订流程包括冻结预算、订机票、订酒店、发送通知、生成报销单和归档六个步骤；酒店预订失败后应释放预算并取消机票。

## 基线

基线在酒店失败时直接停止，留下预算冻结和机票订单。

## 结果解读

手写 Saga 记录已提交动作并按逆序执行补偿。

## 失败案例

补偿动作也会失败，因此状态应为 compensation_pending 而不是完成。

In [1]:
steps = [{'name':'冻结预算','compensate':'释放预算','result':'ok','reversible':True}, {'name':'预订机票','compensate':'取消机票','result':'ok','reversible':True}, {'name':'预订酒店','compensate':'取消酒店','result':'fail','reversible':True}, {'name':'发送通知','compensate':'发送更正','result':'skip','reversible':False}, {'name':'生成报销单','compensate':'作废报销单','result':'skip','reversible':True}, {'name':'归档行程','compensate':'标记作废','result':'skip','reversible':True}]  # 构造六个跨服务出差流程步骤及可补偿语义。
print('Saga 步骤:', steps)  # 输出每步动作、补偿动作、结果与可逆属性。
print('教学说明：所有预订均为离线模拟；真实外部供应商可能只能异步确认取消。')  # 说明 Saga 的现实一致性边界。

Saga 步骤: [{'name': '冻结预算', 'compensate': '释放预算', 'result': 'ok', 'reversible': True}, {'name': '预订机票', 'compensate': '取消机票', 'result': 'ok', 'reversible': True}, {'name': '预订酒店', 'compensate': '取消酒店', 'result': 'fail', 'reversible': True}, {'name': '发送通知', 'compensate': '发送更正', 'result': 'skip', 'reversible': False}, {'name': '生成报销单', 'compensate': '作废报销单', 'result': 'skip', 'reversible': True}, {'name': '归档行程', 'compensate': '标记作废', 'result': 'skip', 'reversible': True}]
教学说明：所有预订均为离线模拟；真实外部供应商可能只能异步确认取消。


In [2]:
unsafe_committed = [step['name'] for step in steps if step['result'] == 'ok']  # 记录基线在失败前已经产生的副作用。
print('直接停止后残留副作用:', unsafe_committed)  # 输出预算冻结和机票预订仍被保留。
print('基线问题：流程失败不等于前面本地事务自动回滚。')  # 强调跨服务无全局事务。

直接停止后残留副作用: ['冻结预算', '预订机票']
基线问题：流程失败不等于前面本地事务自动回滚。


In [3]:
def run_saga(flow, compensation_fail=None):  # 定义记录提交并逆序补偿的手写 Saga 状态机。
    committed = []  # 初始化已经成功提交的本地事务栈。
    compensations = []  # 初始化实际执行的补偿动作日志。
    for step in flow:  # 按业务顺序执行每个本地事务。
        if step['result'] == 'ok':  # 处理成功的可观察本地提交。
            committed.append(step)  # 将成功步骤压入待补偿栈。
            continue  # 继续执行下一个步骤。
        if step['result'] == 'skip':  # 处理尚未执行到的后续步骤。
            break  # 流程在此前失败后不再执行后缀。
        for done in reversed(committed):  # 从最近成功步骤开始逆序补偿。
            if done['reversible']:  # 只对有定义补偿语义的步骤执行补偿。
                status = 'failed' if done['compensate'] == compensation_fail else 'done'  # 模拟补偿自身可能失败。
                compensations.append((done['compensate'], status))  # 落账补偿动作与结果。
        pending = any(status == 'failed' for _, status in compensations)  # 检查是否有补偿尚未完成。
        return 'compensation_pending' if pending else 'compensated', committed, compensations  # 返回明确终态而非伪装成功。
    return 'completed', committed, compensations  # 所有步骤成功时返回正常完成。

In [4]:
status, committed, compensations = run_saga(steps)  # 对酒店失败事件执行逆序 Saga 补偿。
print('流程状态:', status)  # 输出 Saga 的最终业务状态。
print('已提交动作:', [step['name'] for step in committed])  # 输出失败前真实产生的副作用。
print('补偿日志:', compensations)  # 输出取消机票和释放预算的逆序补偿结果。
print('结果解读：酒店失败后不会发送通知，已提交的预算和机票由明确补偿动作处理。')  # 解释状态机为何防止残留副作用。

流程状态: compensated
已提交动作: ['冻结预算', '预订机票']
补偿日志: [('取消机票', 'done'), ('释放预算', 'done')]
结果解读：酒店失败后不会发送通知，已提交的预算和机票由明确补偿动作处理。


In [5]:
pending_status, _, pending_log = run_saga(steps, compensation_fail='取消机票')  # 模拟外部航司取消也失败的补偿反例。
print('失败案例：补偿状态=', pending_status, '，补偿日志=', pending_log)  # 展示补偿失败必须暴露为 pending。
print('生产差距：需持久化 correlation id、幂等键、重试/退避、供应商回调、人工队列和不可逆动作的更正策略。')  # 说明生产 Saga 编排器要求。

失败案例：补偿状态= compensation_pending ，补偿日志= [('取消机票', 'failed'), ('释放预算', 'done')]
生产差距：需持久化 correlation id、幂等键、重试/退避、供应商回调、人工队列和不可逆动作的更正策略。


In [6]:
assert status == 'compensated'  # 验证酒店失败后可逆步骤均已得到补偿。
assert compensations == [('取消机票','done'), ('释放预算','done')]  # 验证补偿顺序严格与提交顺序相反。
assert pending_status == 'compensation_pending'  # 验证补偿失败不会被错误宣布完成。